# Assignment 2 – Smart College Campus Assistant (Multi-Agent AI with CrewAI)


**Scenario chosen:** *NITTE inter-college technical fest* – the assistant plans schedules, venues, budget, guest travel, weather/holiday risks, checklists and student announcements.

## Setup note (how to run this notebook)
1. Open the notebook in **Google Colab** (Runtime → Run all works from top to bottom).
2. Open the 🔑 **Secrets** tab (left sidebar) and add three secrets, then switch **Notebook access ON** for each:
   - `NEXUS_API_KEY` – your Nexus API key
   - `NEXUS_BASE_URL` – the Nexus OpenAI-compatible endpoint URL given in class (e.g. `https://.../v1`)
   - `NEXUS_MODEL` – the model name offered on Nexus
3. Run the cells in order. The first cell installs packages (this takes ~1–2 minutes; if Colab asks to restart the session after installation, restart and continue from the next cell).
4. The two external APIs used (Open-Meteo, Nager.Date) are **free and need no key**.
5. **No keys are stored in this notebook** – they are read at runtime from Colab Secrets only.

## Architecture
```
User request
   │
   ▼
[Event Planner] ──► [Logistics & Guest Coordinator] ──► [Communications Officer] ──► [Quality Reviewer] ──► ONE final response
 venue, schedule,       weather API, travel distance,        LLM announcement tool          merges + checks
 budget, holiday API    preparation checklist                                               everything
```

## 1. Installation
Installs CrewAI (the only agent framework used) and `requests` for the API tools. CrewAI is the base of the whole project; `requests` is used by the external API tools.

In [ ]:
!pip install -q crewai requests nest_asyncio

In [ ]:
import os, json, math, time, datetime, re, traceback
import requests
import nest_asyncio

# Patch the event loop to allow nested execution of asyncio (critical for CrewAI in Colab)
nest_asyncio.apply()

# silence CrewAI telemetry
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from openai import OpenAI

from google.colab import userdata
NEXUS_API_KEY  = userdata.get("NEXUS_API_KEY")
NEXUS_BASE_URL = userdata.get("NEXUS_BASE_URL")

# CRITICAL: Switch model to nova-micro
NEXUS_MODEL    = "nova-micro"

# CRITICAL: Overwrite global environment variables to prevent CrewAI and OpenAI from falling back to incorrect keys
os.environ["OPENAI_API_KEY"] = NEXUS_API_KEY
os.environ["OPENAI_API_BASE"] = NEXUS_BASE_URL

# CrewAI's LLM wrapper pointed at Nexus
llm = LLM(
    model=f"openai/{NEXUS_MODEL}",
    base_url=NEXUS_BASE_URL,
    api_key=NEXUS_API_KEY,
    temperature=0.3,
)

# plain client for the custom LLM tool with explicit keys
nexus_client = OpenAI(api_key=NEXUS_API_KEY, base_url=NEXUS_BASE_URL)

print("Nexus configured. Model:", NEXUS_MODEL, "| endpoint set:", bool(NEXUS_BASE_URL))

Nexus configured. Model: nova-micro | endpoint set: True


### 2.1 Quick Nexus connectivity test
A one-line call to confirm that the Nexus API works *before* building agents. If this fails, check the three secrets.

In [ ]:
# Diagnostic connectivity test using configured environment variables
print(f"Testing connection to: {NEXUS_BASE_URL}")
print(f"Using model: {NEXUS_MODEL}")

try:
    test = nexus_client.chat.completions.create(
        model=NEXUS_MODEL,
        messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        max_tokens=10,
    )
    print("Nexus replied successfully:", test.choices[0].message.content)
except Exception as e:
    print(f"Connection failed! Error: {e}")
    print("\nTip: Verify that NEXUS_API_KEY and NEXUS_BASE_URL are correctly set in your Colab Secrets and match your assigned credentials.")

Testing connection to: https://nexusapi.navigatelabs.ai
Using model: nova-micro
Nexus replied successfully: ready


In [ ]:
# Re-testing the connection with the active client variables
try:
    test = nexus_client.chat.completions.create(
        model=NEXUS_MODEL,
        messages=[{"role": "user", "content": "Reply with the single word: Success!"}],
        max_tokens=10,
    )
    print("Nexus Connectivity Test:", test.choices[0].message.content.strip())
except Exception as e:
    print(f"Connection test failed: {e}")

Nexus Connectivity Test: Victory!


In [ ]:
# List all models available on the Nexus endpoint using the configured API key
try:
    models = nexus_client.models.list()
    print("Available models:")
    for model in models:
        print(f"- {model.id}")
except Exception as e:
    print(f"Failed to retrieve models: {e}")

Available models:
- gemini-2.5-flash
- gpt-4.1-nano
- gpt-4o-mini-tts
- llama3-8b-instruct
- mistral-7b-instruct
- nano-banana
- nova-micro
- text-embedding-3-small
- whisper-1


## 3. Testing the external APIs separately (before integration)
The assignment asks us to test the API on its own first. I use two **free, key-less** APIs:

| API | Used for | Endpoint |
|---|---|---|
| **Open-Meteo** (geocoding + forecast) | weather on the event day, and city coordinates for travel distance | `geocoding-api.open-meteo.com`, `api.open-meteo.com` |
| **Nager.Date** | Indian public holidays – to avoid scheduling the fest on a holiday | `date.nager.at` |

**Output:** raw JSON snippets proving both APIs respond.

In [ ]:
# --- Open-Meteo geocoding ---
g = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                 params={"name": "Mangaluru", "count": 1}, timeout=10)
print("Geocoding status:", g.status_code)
place = g.json()["results"][0]
print({k: place[k] for k in ("name", "latitude", "longitude", "country", "admin1")})

# --- Open-Meteo forecast (next 3 days) ---
w = requests.get("https://api.open-meteo.com/v1/forecast", params={
    "latitude": place["latitude"], "longitude": place["longitude"],
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
    "forecast_days": 3, "timezone": "auto"}, timeout=10)
print("Forecast status:", w.status_code)
print(json.dumps(w.json()["daily"], indent=1))

# --- Nager.Date public holidays (With robust error handling) ---
year = datetime.date.today().year
h = requests.get(f"https://date.nager.at/api/v3/PublicHolidays/{year}/IN", timeout=10)
print("Holiday status:", h.status_code)

if h.status_code == 200:
    try:
        holidays_data = h.json()
        print("Holidays returned:", len(holidays_data))
        print(holidays_data[:2])
    except Exception as e:
        print("Failed to parse JSON. Raw response:", h.text[:500])
else:
    print(f"API request failed with status code {h.status_code}. Raw response:", h.text[:500])

Geocoding status: 200
{'name': 'Mangaluru', 'latitude': 12.91723, 'longitude': 74.85603, 'country': 'India', 'admin1': 'Karnataka'}
Forecast status: 200
{
 "time": [
  "2026-10-26",
  "2026-10-27",
  "2026-10-28"
 ],
 "temperature_2m_max": [
  32.0,
  31.1,
  30.6
 ],
 "temperature_2m_min": [
  24.5,
  22.2,
  22.4
 ],
 "precipitation_probability_max": [
  0,
  0,
  16
 ]
}
Holiday status: 204
API request failed with status code 204. Raw response: 


## 4. Tool development (8 tools)
Every tool is a Python function decorated with `@tool` so CrewAI agents can call it. Each tool:
- validates its inputs and **never raises** – it returns a readable `ERROR:` string instead, so one failing tool cannot crash the crew;
- records itself in `TOOL_LOG`, which lets us **prove every tool was actually used** during execution.

| # | Tool | Type | Agent |
|---|---|---|---|
| 1 | `venue_lookup` | data lookup | Event Planner |
| 2 | `holiday_checker` | **external API** (Nager.Date) | Event Planner |
| 3 | `schedule_builder` | Python logic | Event Planner |
| 4 | `budget_calculator` | calculation | Event Planner |
| 5 | `weather_forecast` | **external API** (Open-Meteo) | Logistics Coordinator |
| 6 | `travel_distance` | **external API** + haversine calculation | Logistics Coordinator |
| 7 | `checklist_generator` | Python logic | Logistics Coordinator |
| 8 | `announcement_writer` | **LLM tool** (via Nexus) | Communications Officer |

### 4.1 Shared helpers and tool-usage log
`TOOL_LOG` stores every tool call (name, inputs, a short output preview). `logged()` is a small helper each tool calls on exit. `geocode()` is reused by the weather and distance tools.

In [ ]:
TOOL_LOG = []   # one entry per tool call - used for observations and the "all tools used" check

def logged(name, args, output):
    TOOL_LOG.append({"tool": name, "args": args, "output_preview": str(output)[:160].replace("\n", " ")})
    return output

def geocode(city):
    """Return (lat, lon, display_name) for a city using the Open-Meteo geocoding API."""
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                     params={"name": city, "count": 1}, timeout=10)
    r.raise_for_status()
    res = r.json().get("results")
    if not res:
        raise ValueError(f"city '{city}' not found")
    p = res[0]
    return p["latitude"], p["longitude"], f"{p['name']}, {p.get('admin1', '')}, {p.get('country', '')}"

def parse_date(s):
    return datetime.datetime.strptime(s.strip(), "%Y-%m-%d").date()

### 4.2 Tool 1 – `venue_lookup` (data lookup)
Looks up a small in-code campus venue database and returns the venues that fit the expected crowd (and AV needs).
**Why:** the planner must pick real venues with enough capacity. **Output:** a list of suitable venues with capacity and facilities.

In [ ]:
CAMPUS_VENUES = {
    "Main Auditorium":     {"capacity": 1000, "av": True,  "facilities": "stage, AC, projector, sound system"},
    "Seminar Hall A":      {"capacity": 250,  "av": True,  "facilities": "AC, projector, mics"},
    "Seminar Hall B":      {"capacity": 150,  "av": True,  "facilities": "AC, projector"},
    "CS Lab Complex":      {"capacity": 120,  "av": True,  "facilities": "120 PCs, LAN, projector"},
    "Open Air Theatre":    {"capacity": 1500, "av": False, "facilities": "open stage, floodlights"},
    "Central Quadrangle":  {"capacity": 2000, "av": False, "facilities": "open ground, stalls space"},
    "Canteen Courtyard":   {"capacity": 300,  "av": False, "facilities": "seating, food stalls"},
}

@tool("venue_lookup")
def venue_lookup(expected_attendees: int, need_av: bool = True) -> str:
    """Find campus venues that can hold the expected number of attendees.
    Args: expected_attendees (int) - people expected; need_av (bool) - True if projector/sound is required."""
    try:
        n = int(expected_attendees)
        if n <= 0:
            return logged("venue_lookup", locals(), "ERROR: expected_attendees must be a positive number")
        fits = [f"- {name}: capacity {v['capacity']}, AV={'yes' if v['av'] else 'no'}, {v['facilities']}"
                for name, v in CAMPUS_VENUES.items()
                if v["capacity"] >= n and (v["av"] or not need_av)]
        out = ("Suitable venues:\n" + "\n".join(fits)) if fits else \
              "No single venue fits; split the event across multiple venues (e.g. Main Auditorium + Seminar Halls)."
        return logged("venue_lookup", {"expected_attendees": n, "need_av": need_av}, out)
    except Exception as e:
        return logged("venue_lookup", {}, f"ERROR: {e}")

### 4.3 Tool 2 – `holiday_checker` (external API – Nager.Date)
Calls the live Nager.Date API for Indian public holidays and tells the planner whether the proposed date (or the days around it) clashes with a holiday.
**Why:** a fest on a holiday has low attendance. **Output:** a clash / no-clash message built from the *actual* API response; on API failure it returns a safe fallback message.

In [ ]:
@tool("holiday_checker")
def holiday_checker(event_date: str, country_code: str = "IN") -> str:
    """Check whether a date (YYYY-MM-DD) is a public holiday, or close to one, using the Nager.Date API.
    Args: event_date - date as YYYY-MM-DD; country_code - ISO country code, default IN."""
    try:
        d = parse_date(event_date)
    except ValueError:
        return logged("holiday_checker", {}, "ERROR: date must be in YYYY-MM-DD format")
    weekday = d.strftime("%A")
    try:
        r = requests.get(f"https://date.nager.at/api/v3/PublicHolidays/{d.year}/{country_code}", timeout=10)
        if r.status_code != 200 or not r.text.strip():
            out = (f"{event_date} is a {weekday}. Holiday data is not available for {country_code} "
                   "(API returned no data). Holiday check UNVERIFIED - proceed and flag this in the plan.")
            return logged("holiday_checker", {"event_date": event_date}, out)
        near = []
        for h in r.json():
            gap = (parse_date(h["date"]) - d).days
            if abs(gap) <= 3:
                near.append(f"{h['name']} on {h['date']} ({'same day' if gap == 0 else f'{gap:+d} days'})")
        if near:
            out = f"{event_date} is a {weekday}. HOLIDAY ALERT within +/-3 days: " + "; ".join(near)
        else:
            out = f"{event_date} is a {weekday}. No public holiday within +/-3 days - date looks safe."
        if weekday == "Sunday":
            out += " Note: it is a Sunday."
        return logged("holiday_checker", {"event_date": event_date}, out)
    except Exception as e:
        return logged("holiday_checker", {}, f"ERROR: holiday API unavailable ({type(e).__name__}). "
                                             "Proceed without holiday data and flag it as unverified.")

### 4.4 Tool 3 – `schedule_builder` (Python logic)
Builds a time-slotted agenda from a start time, end time and a comma-separated list of sessions. When the day spans lunchtime it inserts a **lunch break (13:00–14:00)** and a **15-minute tea break** after the first afternoon session, sizes the morning and afternoon slots separately so the last session ends **exactly at the end time**, and returns an `ERROR:` if the sessions do not fit.
**Output:** a formatted timetable.

In [ ]:
def _fmt(m):
    return f"{m // 60:02d}:{m % 60:02d}"

@tool("schedule_builder")
def schedule_builder(start_time: str, end_time: str, sessions: str) -> str:
    """Create a one-day timetable.
    Args: start_time 'HH:MM'; end_time 'HH:MM'; sessions - comma-separated session names in order."""
    try:
        names = [s.strip() for s in sessions.split(",") if s.strip()]
        if not names:
            return logged("schedule_builder", {}, "ERROR: provide at least one session name")
        sh, sm = map(int, start_time.split(":")); eh, em = map(int, end_time.split(":"))
        start, end = sh * 60 + sm, eh * 60 + em
        if end <= start:
            return logged("schedule_builder", {}, "ERROR: end_time must be after start_time")
        lunch_s, lunch_e, tea = 13 * 60, 14 * 60, 15
        has_lunch = start < lunch_s and end > lunch_e and len(names) >= 2
        rows = []
        if has_lunch:
            est = ((end - start) - (lunch_e - lunch_s) - tea) // len(names)
            k = min(max(1, round((lunch_s - start) / max(est, 1))), len(names) - 1)   # sessions before lunch
            slot_a = (lunch_s - start) // k                                        # morning slot length
            slot_b = (end - lunch_e - tea) // (len(names) - k)                     # afternoon slot length
            if min(slot_a, slot_b) < 20:
                return logged("schedule_builder", {}, f"ERROR: too many sessions ({len(names)}) for the time window")
            t = start
            for nm in names[:k]:
                rows.append(f"{_fmt(t)}-{_fmt(t + slot_a)}  {nm}"); t += slot_a
            rows.append(f"{_fmt(lunch_s)}-{_fmt(lunch_e)}  LUNCH BREAK"); t = lunch_e
            for i, nm in enumerate(names[k:]):
                rows.append(f"{_fmt(t)}-{_fmt(t + slot_b)}  {nm}"); t += slot_b
                if i == 0 and len(names) - k > 1:
                    rows.append(f"{_fmt(t)}-{_fmt(t + tea)}  TEA BREAK"); t += tea
            slot = f"{slot_a}/{slot_b}"
        else:
            slot_n = (end - start) // len(names)
            if slot_n < 20:
                return logged("schedule_builder", {}, f"ERROR: too many sessions ({len(names)}) for the time window")
            t = start
            for nm in names:
                rows.append(f"{_fmt(t)}-{_fmt(t + slot_n)}  {nm}"); t += slot_n
            slot = str(slot_n)
        out = f"Timetable ({len(names)} sessions, slot lengths {slot} min):\n" + "\n".join(rows)
        return logged("schedule_builder", {"start": start_time, "end": end_time, "n_sessions": len(names)}, out)
    except Exception as e:
        return logged("schedule_builder", {}, f"ERROR: {e}. Use HH:MM times and comma-separated sessions.")

### 4.5 Tool 4 – `budget_calculator` (calculation)
Computes an estimated budget from the number of attendees using a per-head rate card for catering, certificates, printing and security, plus fixed items and a **10 % contingency**.
**Output:** an itemised budget table and the grand total in ₹.

In [ ]:
@tool("budget_calculator")
def budget_calculator(attendees: int, num_guests: int = 1, prize_pool: int = 30000) -> str:
    """Estimate event budget in INR.
    Args: attendees (int) participants; num_guests (int) invited guests/speakers; prize_pool (int) total prize money."""
    try:
        a, g, p = int(attendees), int(num_guests), int(prize_pool)
        if a <= 0:
            return logged("budget_calculator", {}, "ERROR: attendees must be positive")
        items = {
            "Snacks & lunch (Rs 150/head)":        a * 150,
            "Certificates & printing (Rs 25/head)": a * 25,
            "Volunteer T-shirts & ID (Rs 120/head x 10%)": int(a * 0.10) * 120,
            "Guest honorarium & gifts (Rs 5,000/guest)": g * 5000,
            "Guest accommodation (Rs 3,500/guest)": g * 3500,
            "Sound, stage & decoration (fixed)":    25000,
            "Prize pool":                           p,
        }
        sub = sum(items.values()); cont = round(sub * 0.10); total = sub + cont
        lines = [f"- {k}: Rs {v:,}" for k, v in items.items()]
        out = "Budget estimate:\n" + "\n".join(lines) + f"\nSubtotal: Rs {sub:,}\nContingency (10%): Rs {cont:,}\nGRAND TOTAL: Rs {total:,}"
        return logged("budget_calculator", {"attendees": a, "guests": g, "prize_pool": p}, out)
    except Exception as e:
        return logged("budget_calculator", {}, f"ERROR: {e}")

### 4.6 Tool 5 – `weather_forecast` (external API – Open-Meteo)
Geocodes the city, then requests the **daily forecast** for the event date. If the date is beyond the 16-day forecast window, it falls back to the next 3 days and says so clearly (no invented data).
**Why:** outdoor stages and guest travel depend on weather. **Output:** max/min temperature and rain probability from the real API response, with an advice line.

In [ ]:
@tool("weather_forecast")
def weather_forecast(city: str, event_date: str) -> str:
    """Get the weather forecast for a city on a date (YYYY-MM-DD) from the Open-Meteo API.
    Args: city - city name; event_date - YYYY-MM-DD."""
    try:
        lat, lon, label = geocode(city)
        d = parse_date(event_date)
        base = {"latitude": lat, "longitude": lon, "timezone": "auto",
                "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max"}
        note = ""
        r = requests.get("https://api.open-meteo.com/v1/forecast",
                         params={**base, "start_date": d.isoformat(), "end_date": d.isoformat()}, timeout=10)
        if r.status_code != 200:     # date outside the forecast window
            r = requests.get("https://api.open-meteo.com/v1/forecast", params={**base, "forecast_days": 3}, timeout=10)
            r.raise_for_status()
            note = f"(Event date {event_date} is outside the 16-day forecast window; showing the next 3 days instead.) "
        daily = r.json()["daily"]
        rows = [f"{day}: {lo}-{hi} C, rain chance {rain}%"
                for day, hi, lo, rain in zip(daily["time"], daily["temperature_2m_max"],
                                             daily["temperature_2m_min"], daily["precipitation_probability_max"])]
        worst = max(x or 0 for x in daily["precipitation_probability_max"])
        advice = "Rain likely - keep an indoor backup venue ready." if worst >= 50 else "Rain risk is low - outdoor events are feasible."
        out = f"{note}Weather for {label}: " + " | ".join(rows) + f". Advice: {advice}"
        return logged("weather_forecast", {"city": city, "date": event_date}, out)
    except Exception as e:
        return logged("weather_forecast", {}, f"ERROR: weather unavailable ({type(e).__name__}: {e}). "
                                              "State that weather could not be verified and plan an indoor backup.")

### 4.7 Tool 6 – `travel_distance` (API + calculation)
Geocodes two cities with the Open-Meteo API and computes the great-circle (**haversine**) distance, then suggests a travel mode and an approximate travel time for an arriving guest (road distance ≈ 1.3 × straight line).
**Output:** distance in km, suggested mode and estimated travel time.

In [ ]:
@tool("travel_distance")
def travel_distance(from_city: str, to_city: str = "Mangaluru") -> str:
    """Estimate distance and travel time for a guest travelling between two cities.
    Args: from_city - guest's city; to_city - campus city (default Mangaluru)."""
    try:
        la1, lo1, n1 = geocode(from_city)
        la2, lo2, n2 = geocode(to_city)
        R = 6371
        p1, p2 = math.radians(la1), math.radians(la2)
        a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lo2 - lo1) / 2) ** 2
        km = round(2 * R * math.asin(math.sqrt(a)))
        road = round(km * 1.3)
        if km < 150:
            mode, hrs = "road (cab/car)", road / 50
        elif km < 600:
            mode, hrs = "train or overnight bus/cab", road / 55
        else:
            mode, hrs = "flight (then airport cab)", km / 700 + 3
        out = (f"{n1} -> {n2}: straight-line {km} km, approx road {road} km. "
               f"Suggested mode: {mode}. Approx travel time: {hrs:.1f} hours.")
        return logged("travel_distance", {"from": from_city, "to": to_city}, out)
    except Exception as e:
        return logged("travel_distance", {}, f"ERROR: could not compute distance ({type(e).__name__}: {e}). "
                                             "Ask the organiser to confirm the guest's travel mode manually.")

### 4.8 Tool 7 – `checklist_generator` (Python logic)
Generates a **preparation checklist with real deadlines** counted backwards from the event date (T-14 days, T-7, T-3, T-1, event day). If the event is closer than a deadline, that item is marked **OVERDUE / do immediately**.
**Output:** dated checklist with owners.

In [ ]:
@tool("checklist_generator")
def checklist_generator(event_date: str, event_type: str = "technical fest") -> str:
    """Generate a dated preparation checklist counting back from the event date.
    Args: event_date - YYYY-MM-DD; event_type - e.g. 'technical fest', 'guest lecture'."""
    try:
        d = parse_date(event_date)
        today = datetime.date.today()
        plan = [
            (14, "Finalise venues and get principal's permission", "Event Committee"),
            (14, "Open registrations and confirm guests/judges", "Registration Team"),
            (10, "Book guest travel and accommodation", "Hospitality Team"),
            (7,  "Print posters, certificates, ID cards; confirm catering", "Logistics Team"),
            (5,  "Test AV, internet/LAN and lab systems", "Technical Team"),
            (3,  "Brief volunteers; publish final schedule", "Volunteer Coordinator"),
            (1,  "Venue decoration, signage, mock run-through", "All Teams"),
            (0,  "Registration desk, live coordination, feedback forms", "All Teams"),
        ]
        rows = []
        for days_before, task, owner in plan:
            due = d - datetime.timedelta(days=days_before)
            flag = "  <-- OVERDUE / do immediately" if due < today else ""
            rows.append(f"- [T-{days_before}] {due.isoformat()}: {task} (Owner: {owner}){flag}")
        out = f"Checklist for {event_type} on {event_date} (today = {today}):\n" + "\n".join(rows)
        return logged("checklist_generator", {"event_date": event_date, "type": event_type}, out)
    except ValueError:
        return logged("checklist_generator", {}, "ERROR: event_date must be YYYY-MM-DD")
    except Exception as e:
        return logged("checklist_generator", {}, f"ERROR: {e}")

### 4.9 Tool 8 – `announcement_writer` (LLM tool via Nexus)
A **custom LLM tool**: it takes the final event summary and writes a student-friendly announcement by calling the **Nexus API** directly (not any other provider).
**Why:** drafting engaging copy is a language task, not Python logic. **Output:** a ready-to-post announcement. If the Nexus call fails it returns a plain template so the workflow still completes.

In [ ]:
@tool("announcement_writer")
def announcement_writer(event_summary: str, audience: str = "students", tone: str = "enthusiastic") -> str:
    """Write a campus announcement using an LLM (Nexus).
    Args: event_summary - key facts (name, date, venue, timings, how to register); audience - who it is for; tone - style."""
    try:
        resp = nexus_client.chat.completions.create(
            model=NEXUS_MODEL,
            messages=[
                {"role": "system", "content": "You write short, clear college announcements with a catchy title, "
                                              "key details as bullet points and a call to action. Max 120 words. "
                                              "Use ONLY facts supplied; do not invent details."},
                {"role": "user", "content": f"Audience: {audience}\nTone: {tone}\nFacts:\n{event_summary}"},
            ],
            temperature=0.5, max_tokens=400,
        )
        return logged("announcement_writer", {"audience": audience, "tone": tone}, resp.choices[0].message.content.strip())
    except Exception as e:
        fallback = f"ANNOUNCEMENT (template - LLM unavailable: {type(e).__name__})\n{event_summary}\nRegister now at the department office."
        return logged("announcement_writer", {}, fallback)

### 4.10 Unit-testing each tool directly (no agents yet)
Before giving tools to agents, I call each one by hand. This shows all 8 tools work and returns their real output (including the live API responses). `TOOL_LOG` is cleared afterwards.

In [ ]:
def call_tool(t, **kw):
    # CrewAI wraps the function in a Tool object; .func is the original Python function
    return t.func(**kw)

fest_date = (datetime.date.today() + datetime.timedelta(days=10)).isoformat()
print("Test date:", fest_date, "\n")
print("1.", call_tool(venue_lookup, expected_attendees=400), "\n")
print("2.", call_tool(holiday_checker, event_date=fest_date), "\n")
print("3.", call_tool(schedule_builder, start_time="09:00", end_time="17:00",
                       sessions="Inauguration,Hackathon,Paper Presentation,Robo Race,Quiz,Valedictory"), "\n")
print("4.", call_tool(budget_calculator, attendees=400, num_guests=2, prize_pool=40000), "\n")
print("5.", call_tool(weather_forecast, city="Mangaluru", event_date=fest_date), "\n")
print("6.", call_tool(travel_distance, from_city="Bengaluru", to_city="Mangaluru"), "\n")
print("7.", call_tool(checklist_generator, event_date=fest_date), "\n")
print("8.", call_tool(announcement_writer, event_summary="TechNova 2026, 25 Oct, Main Auditorium, 9 AM-5 PM, register at tech.nitte.edu"), "\n")
print("Tools exercised:", sorted({e['tool'] for e in TOOL_LOG}))
TOOL_LOG.clear()

Test date: 2026-11-05 

1. Suitable venues:
- Main Auditorium: capacity 1000, AV=yes, stage, AC, projector, sound system 

2. ERROR: date must be in YYYY-MM-DD format 

3. Timetable (6 sessions, slot lengths 60/82 min):
09:00-10:00  Inauguration
10:00-11:00  Hackathon
11:00-12:00  Paper Presentation
12:00-13:00  Robo Race
13:00-14:00  LUNCH BREAK
14:00-15:22  Quiz
15:22-15:37  TEA BREAK
15:37-16:59  Valedictory 

4. Budget estimate:
- Snacks & lunch (Rs 150/head): Rs 60,000
- Certificates & printing (Rs 25/head): Rs 10,000
- Volunteer T-shirts & ID (Rs 120/head x 10%): Rs 4,800
- Guest honorarium & gifts (Rs 5,000/guest): Rs 10,000
- Guest accommodation (Rs 3,500/guest): Rs 7,000
- Sound, stage & decoration (fixed): Rs 25,000
- Prize pool: Rs 40,000
Subtotal: Rs 156,800
Contingency (10%): Rs 15,680
GRAND TOTAL: Rs 172,480 

5. Weather for Mangaluru, Karnataka, India: 2026-11-05: 23.8-29.8 C, rain chance 79%. Advice: Rain likely - keep an indoor backup venue ready. 

6. Bengaluru, Karna

## 5. Agent creation (4 agents)
Each agent has its own **role, goal and backstory** and a *specific responsibility*. Tools are assigned only to the agents that need them. `allow_delegation=False` keeps the workflow strictly sequential and predictable.

| Agent | Responsibility | Tools |
|---|---|---|
| **Event Planner** | venue, date safety, agenda, budget | venue_lookup, holiday_checker, schedule_builder, budget_calculator |
| **Logistics & Guest Coordinator** | weather risk, guest travel, preparation checklist | weather_forecast, travel_distance, checklist_generator |
| **Communications Officer** | student announcement | announcement_writer |
| **Quality Reviewer** | merge, check consistency, produce the one final answer | (none – reasons over previous outputs) |

In [ ]:
planner = Agent(
    role="Event Planner",
    goal="Produce a feasible event plan: suitable venue, holiday-safe date, a timed agenda and a realistic budget.",
    backstory="A veteran faculty coordinator who has run NITTE's technical fests for ten years. Obsessed with "
              "capacity limits, clash-free dates and budgets that never overshoot. Always uses tools instead of guessing.",
    tools=[venue_lookup, holiday_checker, schedule_builder, budget_calculator],
    llm=llm, allow_delegation=False, verbose=False, max_iter=10,
)

logistics = Agent(
    role="Logistics & Guest Coordinator",
    goal="Handle guest travel, weather risks and a dated preparation checklist so nothing is missed on the event day.",
    backstory="An operations lead who has hosted hundreds of visiting speakers. Checks the weather and travel time "
              "for every guest and turns plans into a deadline-driven checklist.",
    tools=[weather_forecast, travel_distance, checklist_generator],
    llm=llm, allow_delegation=False, verbose=False, max_iter=10,
)

comms = Agent(
    role="Communications Officer",
    goal="Write a clear, accurate and engaging announcement for students using only verified event facts.",
    backstory="The student-council media head with a flair for catchy yet precise campus notices. "
              "Never invents details - works only from what the planner and logistics team confirmed.",
    tools=[announcement_writer],
    llm=llm, allow_delegation=False, verbose=False, max_iter=6,
)

reviewer = Agent(
    role="Quality Reviewer",
    goal="Combine all team outputs into ONE consistent, well-structured final response and flag risks or assumptions.",
    backstory="A meticulous registrar who proof-reads every circular. Checks that dates, venues, timings and costs agree "
              "across all sections and clearly lists assumptions and warnings.",
    tools=[], llm=llm, allow_delegation=False, verbose=False, max_iter=4,
)
print("Agents recreated with active LLM settings:", [a.role for a in (planner, logistics, comms, reviewer)])

Agents recreated with active LLM settings: ['Event Planner', 'Logistics & Guest Coordinator', 'Communications Officer', 'Quality Reviewer']


In [ ]:
print("--- CrewAI LLM Object Verification ---")
print("LLM Model ID:", llm.model)
print("LLM Base URL:", llm.base_url)
print("Active Model from Config:", NEXUS_MODEL)

print("\n--- Agent-Specific LLM Verification ---")
for agent in [planner, logistics, comms, reviewer]:
    print(f"{agent.role:30} uses model: {getattr(agent.llm, 'model', 'default')}")

--- CrewAI LLM Object Verification ---
LLM Model ID: nova-micro
LLM Base URL: https://nexusapi.navigatelabs.ai
Active Model from Config: nova-micro

--- Agent-Specific LLM Verification ---
Event Planner                  uses model: nova-micro
Logistics & Guest Coordinator  uses model: nova-micro
Communications Officer         uses model: nova-micro
Quality Reviewer               uses model: nova-micro


## 6. Task definition with chained context
Each task has a clear **description** and **expected_output**. The `context=[...]` argument is what *chains* the tasks: the Logistics task receives the Planner's output, the Communications task receives both previous outputs, and the Reviewer receives everything.

`{request}` and `{today}` are filled in at run time by `crew.kickoff(inputs=...)`. If the user omits a date or city, the Planner is told to assume safe defaults and **state the assumption** (this is part of the failure handling).

In [ ]:
task_plan = Task(
    description=(
        "USER REQUEST: {request}\nToday's date: {today}\n\n"
        "You are planning a campus event. Extract the event type, expected attendees, date and city from the request. "
        "If the date is missing, assume a date 21 days from today that is a weekday; if attendees are missing assume 300; "
        "if the city is missing assume Mangaluru. List every assumption explicitly.\n"
        "You MUST call these tools (do not guess their results):\n"
        "1. venue_lookup for the expected attendees\n"
        "2. holiday_checker for the event date\n"
        "3. schedule_builder with 6 relevant session names (09:00-17:00 unless the user says otherwise)\n"
        "4. budget_calculator with attendees and number of guests (default 1)"
    ),
    expected_output=("A structured plan with: assumptions, chosen venue (from tool), holiday check result (from tool), "
                     "full timetable (from tool) and itemised budget with grand total (from tool). Dates must be YYYY-MM-DD."),
    agent=planner,
)

task_logistics = Task(
    description=(
        "Using the Event Planner's plan, handle logistics for the user request: {request}\nToday's date: {today}\n"
        "You MUST call these tools:\n"
        "1. weather_forecast for the event city and event date from the plan\n"
        "2. travel_distance from the guest's city to the event city (if no guest city is mentioned, use Bengaluru as an assumption)\n"
        "3. checklist_generator with the event date and event type\n"
        "Report any tool ERROR honestly; never invent weather or distance values."
    ),
    expected_output=("Sections: Weather (real forecast + advice), Guest travel (distance, mode, time, arrival recommendation), "
                     "and the dated preparation checklist. Mention any tool that failed and the fallback taken."),
    agent=logistics, context=[task_plan],
)

task_comms = Task(
    description=(
        "Using the plan and logistics outputs, write the student announcement.\n"
        "First compile a short fact summary (event name, date, venue, timings, weather note, how to register - invent no facts; "
        "use 'to be announced' where unknown). Then call announcement_writer with that summary, audience 'students' "
        "and an enthusiastic tone. Return the tool's announcement."
    ),
    expected_output="A ready-to-post announcement (title, bullet points with real details, call to action, under 120 words).",
    agent=comms, context=[task_plan, task_logistics],
)

task_review = Task(
    description=(
        "Merge the outputs of all previous tasks into ONE final response for the user request: {request}\n"
        "Check that dates, venue, timings and budget are consistent across sections. "
        "Use exactly these headings: 1) Event Overview & Assumptions, 2) Venue & Date Check, 3) Schedule, 4) Budget, "
        "5) Weather & Guest Travel, 6) Preparation Checklist, 7) Student Announcement, 8) Risks & Notes."
    ),
    expected_output="One complete, consistent, well-formatted final response with the 8 headings and no contradictions.",
    agent=reviewer, context=[task_plan, task_logistics, task_comms],
)
tasks = [task_plan, task_logistics, task_comms, task_review]
print("Tasks created:", len(tasks), "| chain: plan -> logistics -> comms -> review")

Tasks created: 4 | chain: plan -> logistics -> comms -> review


## 7. Workflow execution
`run_assistant(request)` is the single entry point:
1. **Validates the request** (empty, too short, or not campus-event related → returns a helpful message, never crashes).
2. Builds a **sequential** `Crew` of the 4 agents and 4 tasks and runs `kickoff`.
3. A `task_callback` records each finished task, so we can show **information sharing between agents**.
4. Everything is wrapped in `try/except`; if the crew itself fails, a clear error report is returned.
5. Returns the one final combined answer plus run metrics.

`show_run()` prints the input, the agent steps, the tools used with outputs, the final response and a tool-coverage check.

In [ ]:
import asyncio
import concurrent.futures

EVENT_WORDS = ["event", "fest", "schedule", "plan", "guest", "speaker", "seminar", "workshop", "hackathon", "competition",
               "announcement", "orientation", "convocation", "farewell", "cultural", "sports", "visit", "lecture", "prepare", "organis", "organiz"]
ALL_TOOLS = ["venue_lookup", "holiday_checker", "schedule_builder", "budget_calculator",
             "weather_forecast", "travel_distance", "checklist_generator", "announcement_writer"]
TASK_LOG = []

def _on_task_done(out):
    TASK_LOG.append({"agent": out.agent, "summary": (out.raw or "")[:300].replace("\n", " ")})

def validate_request(req):
    if not isinstance(req, str) or not req.strip():
        return "Your request is empty. Please describe the campus event you want help with (type, size, date, city)."
    if len(req.strip()) < 12:
        return "Your request is too short to act on. Please add details: what event, roughly how many people, which date?"
    if not any(w in req.lower() for w in EVENT_WORDS):
        return ("I could not recognise a campus-event task in your request. I can plan schedules, guest visits, "
                "announcements and event preparations - please rephrase with the event you need help with.")
    return None

def run_assistant(request):
    import nest_asyncio
    nest_asyncio.apply()

    TOOL_LOG.clear(); TASK_LOG.clear()
    problem = validate_request(request)
    if problem:
        return {"ok": False, "final": problem, "seconds": 0.0}
    t0 = time.time()
    try:
        crew = Crew(agents=[planner, logistics, comms, reviewer], tasks=tasks, process=Process.sequential,
                    task_callback=_on_task_done, verbose=False, memory=False)

        # Define helper to run the coroutine in an isolated event loop on another thread
        def _run_async_kickoff():
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            try:
                return loop.run_until_complete(crew.kickoff_async(inputs={"request": request, "today": datetime.date.today().isoformat()}))
            finally:
                loop.close()

        with concurrent.futures.ThreadPoolExecutor() as executor:
            future = executor.submit(_run_async_kickoff)
            result = future.result()

        return {"ok": True, "final": result.raw, "seconds": round(time.time() - t0, 1)}
    except Exception as e:
        return {"ok": False, "seconds": round(time.time() - t0, 1),
                "final": f"The crew hit an error ({type(e).__name__}: {str(e)[:200]}). No data was lost; please retry or rephrase the request."}

def show_run(title, request):
    print("=" * 100); print(f"TEST: {title}"); print("=" * 100)
    print("INPUT REQUEST:\n ", request, "\n")
    res = run_assistant(request)
    print("-- AGENT STEPS (each task result is passed as context to the next agent) --")
    for i, s in enumerate(TASK_LOG, 1):
        print(f" {i}. {s["agent"]}: {s["summary"][:200]}...")
    print("\n-- TOOLS USED --")
    for e in TOOL_LOG:
        print(f" * {e["tool"]}  args={e["args"]}\n     -> {e["output_preview"]}")
    used = {e["tool"] for e in TOOL_LOG}
    print(f"\n-- TOOL COVERAGE: {len(used)}/{len(ALL_TOOLS)} used; not used: {sorted(set(ALL_TOOLS) - used) or "none"}")
    print("\n-- FINAL RESPONSE --\n" + res["final"])
    print(f"\n-- METRICS: ok={res["ok"]}, time={res["seconds"]}s, tool calls={len(TOOL_LOG)} --\n")
    return res

## 8. Testing (Topic 2)
Three normal test requests with different scenarios, then failure tests. For each: the input, agent steps, tools used and outputs, the final response, and my observations.

### Test 1 – Inter-college technical fest (full planning)

In [ ]:
req1 = ("Plan a one-day inter-college technical fest at NITTE Mangaluru for about 400 participants. "
        "Keep it on " + (datetime.date.today() + datetime.timedelta(days=12)).isoformat() +
        ". Include a hackathon, paper presentation and robo race. Two judges are visiting.")
r1 = show_run("Inter-college technical fest", req1)

TEST: Inter-college technical fest
INPUT REQUEST:
  Plan a one-day inter-college technical fest at NITTE Mangaluru for about 400 participants. Keep it on 2026-11-07. Include a hackathon, paper presentation and robo race. Two judges are visiting. 



-- AGENT STEPS (each task result is passed as context to the next agent) --
 1. Event Planner: Here's the detailed plan for the one-day inter-college technical fest at NITTE Mangaluru:  ### Assumptions: - **Event Type:** One-day inter-college technical fest - **Expected Attendees:** 400 (as pro...
 2. Event Planner: Here's the detailed plan for the one-day inter-college technical fest at NITTE Mangaluru:  ### Assumptions: - **Event Type:** One-day inter-college technical fest - **Expected Attendees:** 400 (as pro...
 3. Logistics & Guest Coordinator: ### Final Answer for the One-Day Inter-College Technical Fest Logistics  **Weather:** - **Forecast for Mangaluru, Karnataka, India on 2026-11-07:** The temperature will range between 22.8°C and 29.3°C...
 4. Logistics & Guest Coordinator: ### Final Answer for the One-Day Inter-College Technical Fest Logistics  **Weather:** - **Forecast for Mangaluru, Karnataka, India on 2026-11-07:** The temperature will range between 22.8°C and 29.3°C...


**Observations – Test 1** *(fill in after running; use the metrics printed above)*
- Were all 8 tools used? (see TOOL COVERAGE line) …
- Did the final answer use the *actual* weather and holiday API results? …
- Did the timetable/budget in the final answer match the tool output exactly? …
- Runtime and anything surprising: …

### Test 2 – Guest speaker visiting from another city

In [ ]:
req2 = ("Organise the visit of a guest speaker, an AI researcher arriving from Hyderabad, who will give a seminar "
        "for 150 students on " + (datetime.date.today() + datetime.timedelta(days=8)).isoformat() +
        " in the morning. Arrange travel info and a checklist.")
r2 = show_run("Guest speaker visit", req2)

TEST: Guest speaker visit
INPUT REQUEST:
  Organise the visit of a guest speaker, an AI researcher arriving from Hyderabad, who will give a seminar for 150 students on 2026-11-03 in the morning. Arrange travel info and a checklist. 



-- AGENT STEPS (each task result is passed as context to the next agent) --
 1. Event Planner: Here's the detailed plan for organizing the visit of the guest speaker, an AI researcher from Hyderabad, for a seminar on 2026-11-03:  ### Assumptions: - **Event Type:** Seminar by a guest speaker - *...
 2. Event Planner: Here's the detailed plan for organizing the visit of the guest speaker, an AI researcher from Hyderabad, for a seminar on 2026-11-03:  ### Assumptions: - **Event Type:** Seminar by a guest speaker - *...
 3. Logistics & Guest Coordinator: Here is the complete logistics plan for organizing the visit of the guest speaker, an AI researcher from Hyderabad, for a seminar on 2026-11-03 in Mangaluru:  ### Weather **Forecast for Mangaluru on 2...
 4. Logistics & Guest Coordinator: Here is the complete logistics plan for organizing the visit of the guest speaker, an AI researcher from Hyderabad, for a seminar on 2026-11-03 in Mangaluru:  ### Weather **Forecast for Mangaluru on 2...


**Observations – Test 2** *(fill in after running)*
- Was the Hyderabad → Mangaluru distance/mode realistic? …
- Did the venue choice match the 150-seat requirement? …
- Did the checklist mark any overdue items correctly? …

### Test 3 – Announcements for a cultural event (different type of request)

In [ ]:
req3 = ("Prepare announcements for students about the annual cultural day for 600 students on " +
        (datetime.date.today() + datetime.timedelta(days=15)).isoformat() +
        " at the Open Air Theatre, with dance, music and drama competitions. Suggest event preparations and coordination activities.")
r3 = show_run("Cultural day announcement + preparations", req3)

TEST: Cultural day announcement + preparations
INPUT REQUEST:
  Prepare announcements for students about the annual cultural day for 600 students on 2026-11-10 at the Open Air Theatre, with dance, music and drama competitions. Suggest event preparations and coordination activities. 



ERROR:crewai.flow.runtime:Error executing listener execute_native_tool: Tool 'holiday_checker' failed during 'USER REQUEST: Prepare announcements for students about the annual cultural day for 600 students on 2026-11-10 at the Open Air Theatre, with dance, music and drama competitions. Suggest event preparations and coordination activities.
Today's date: 2026-10-26

You are planning a campus event. Extract the event type, expected attendees, date and city from the request. If the date is missing, assume a date 21 days from today that is a weekday; if attendees are missing assume 300; if the city is missing assume Mangaluru. List every assumption explicitly.
You MUST call these tools (do not guess their results):
1. venue_lookup for the expected attendees
2. holiday_checker for the event date
3. schedule_builder with 6 relevant session names (09:00-17:00 unless the user says otherwise)
4. budget_calculator with attendees and number of guests (default 1)': Agent 'Event Planner' called 'h

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

-- AGENT STEPS (each task result is passed as context to the next agent) --

-- TOOLS USED --
 * venue_lookup  args={'expected_attendees': 600, 'need_av': True}
     -> Suitable venues: - Main Auditorium: capacity 1000, AV=yes, stage, AC, projector, sound system
 * holiday_checker  args={}
     -> ERROR: date must be in YYYY-MM-DD format
 * holiday_checker  args={}
     -> ERROR: date must be in YYYY-MM-DD format
 * holiday_checker  args={}
     -> ERROR: date must be in YYYY-MM-DD format
 * holiday_checker  args={}
     -> ERROR: date must be in YYYY-MM-DD format
 * holiday_checker  args={}
     -> ERROR: date must be in YYYY-MM-DD format

-- TOOL COVERAGE: 2/8 used; not used: ['announcement_writer', 'budget_calculator', 'checklist_generator', 'schedule_builder', 'travel_distance', 'weather_forecast']

-- FINAL RESPONSE --
The crew hit an error (ToolExecutionFailedError: Tool 'holiday_checker' failed during 'USER REQUEST: Prepare announcements for students about the annual cultural da

**Observations – Test 3** *(fill in after running)*
- How did the open-air venue interact with the weather result (backup plan mentioned)? …
- Was the announcement free of invented facts? …
- Consistency across the 4 agents' outputs: …

### Failure Test A – Missing / incomplete information
The request has **no date, no attendee count, no city and no guest city**. The expected behaviour is that the Planner follows the assumption rules, lists the assumptions, and the system still returns a full answer without crashing.

In [ ]:
r_fail_a = show_run("Failure A: incomplete request", "Plan a fest for our college")

TEST: Failure A: incomplete request
INPUT REQUEST:
  Plan a fest for our college 



-- AGENT STEPS (each task result is passed as context to the next agent) --
 1. Event Planner: Here's the detailed plan for the fest:  **Assumptions:** - Event Type: Not specified, assuming a general technical fest. - Expected Attendees: 300 (default assumption). - Date: 2026-11-06 (21 days fro...
 2. Event Planner: Here's the detailed plan for the fest:  **Assumptions:** - Event Type: Not specified, assuming a general technical fest. - Expected Attendees: 300 (default assumption). - Date: 2026-11-06 (21 days fro...
 3. Logistics & Guest Coordinator: ### Final Answer:  **Weather:** - **Forecast for Mangaluru, Karnataka, India on 2026-11-06:** Temperatures will range from 22.8°C to 29.3°C with a 69% chance of rain. - **Advice:** Given the high like...
 4. Logistics & Guest Coordinator: ### Final Answer:  **Weather:** - **Forecast for Mangaluru, Karnataka, India on 2026-11-06:** Temperatures will range from 22.8°C to 29.3°C with a 69% chance of rain. - **Advice:** Given the high like...


### Failure Test B – Invalid / unclear requests (input validation)
These are caught by `validate_request` *before* any LLM or tool call is made, so no tokens are wasted and nothing crashes.

In [ ]:
for bad in ["", "hi", "what is the capital of France and who won the cricket world cup?"]:
    out = run_assistant(bad)
    print(f"INPUT: {bad!r}\n -> handled gracefully (ok={out['ok']}): {out['final']}\n")

### Failure Test C – External API failure (simulated outage)
I temporarily replace `requests.get` with a function that raises a `ConnectionError`, simulating Open-Meteo and Nager.Date being down. The tools must return `ERROR:` fallback messages (not exceptions) and the reviewer must flag the data as unverified. The original `requests.get` is restored afterwards.

In [ ]:
_real_get = requests.get
def _broken_get(*a, **k):
    raise requests.exceptions.ConnectionError("simulated API outage")

requests.get = _broken_get
try:
    print("Direct tool check while APIs are down:")
    print(" weather :", call_tool(weather_forecast, city="Mangaluru", event_date="2026-12-01"))
    print(" holiday :", call_tool(holiday_checker, event_date="2026-12-01"))
    print(" distance:", call_tool(travel_distance, from_city="Pune"), "\n")
    TOOL_LOG.clear()
    r_fail_c = show_run("Failure C: API outage",
        "Plan a one-day workshop for 200 students on " + (datetime.date.today() + datetime.timedelta(days=14)).isoformat() +
        " with a guest speaker arriving from Pune.")
finally:
    requests.get = _real_get      # always restore the real function
print("requests.get restored:", requests.get is _real_get)

Direct tool check while APIs are down:
 weather : ERROR: weather unavailable (ConnectionError: simulated API outage). State that weather could not be verified and plan an indoor backup.
 holiday : ERROR: holiday API unavailable (ConnectionError). Proceed without holiday data and flag it as unverified.
 distance: ERROR: could not compute distance (ConnectionError: simulated API outage). Ask the organiser to confirm the guest's travel mode manually. 

TEST: Failure C: API outage
INPUT REQUEST:
  Plan a one-day workshop for 200 students on 2026-11-09 with a guest speaker arriving from Pune. 



ERROR:root:Model gemini-2.5-flash not found: Error code: 404 - {'error': {'message': 'litellm.NotFoundError: GeminiException - {\n  "error": {\n    "code": 404,\n    "message": "This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).",\n    "status": "NOT_FOUND"\n  }\n}\n. Received Model Group=gemini-2.5-flash\nAvailable Model Group Fallbacks=None', 'type': 'invalid_request_error', 'param': None, 'code': '404'}}
ERROR:root:OpenAI API call failed: Model gemini-2.5-flash not found: Error code: 404 - {'error': {'message': 'litellm.NotFoundError: GeminiException - {\n  "error": {\n    "code": 404,\n    "message": "This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: Model gemini-2.5-flash not found: Error code: 404 - {'error': {'message': 'litellm.NotFoundError: GeminiException - {\n  "error": {\n    "code": 404,\n    "message": "This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).",\n    "status": "NOT_FOUND"\n  }\n}\n. Received Model Group=gemini-2.5-flash\nAvailable Model Group Fallbacks=None', 'type': 'invalid_request_error', 'param': None, 'code': '404'}}


-- AGENT STEPS (each task result is passed as context to the next agent) --

-- TOOLS USED --

-- TOOL COVERAGE: 0/8 used; not used: ['announcement_writer', 'budget_calculator', 'checklist_generator', 'holiday_checker', 'schedule_builder', 'travel_distance', 'venue_lookup', 'weather_forecast']

-- FINAL RESPONSE --
The crew hit an error (ValueError: Model gemini-2.5-flash not found: Error code: 404 - {'error': {'message': 'litellm.NotFoundError: GeminiException - {\n  "error": {\n    "code": 404,\n    "message": "This model models/gemini-2.5-flas). No data was lost; please retry or rephrase the request.

-- METRICS: ok=False, time=5.5s, tool calls=0 --

requests.get restored: True


**How the application handles failures without crashing**
1. **Missing information →** the task prompts define default assumptions and force the agent to list them.
2. **Invalid / off-topic / empty input →** `validate_request` returns a helpful message before any agent runs.
3. **API failure →** every tool has `try/except` and returns an `ERROR:` string with an instruction (e.g. "flag as unverified"); the agent reads it and reports it honestly instead of fabricating data.
4. **LLM-tool failure →** `announcement_writer` falls back to a plain template.
5. **Crew-level failure →** `run_assistant` catches any exception and returns a readable message.

**Observations – failure tests** *(fill in after running: did the final answer clearly mention the unverified weather/holiday data?)* …

## 9. Results summary
A table generated from the runs above: how many tool calls, whether every tool was used, and time taken.

In [ ]:
# Fetch variables safely from globals; default to a mock dict if they don't exist yet
_r1 = globals().get('r1', {'ok': False, 'seconds': 0.0, 'final': 'Not run yet'})
_r2 = globals().get('r2', {'ok': False, 'seconds': 0.0, 'final': 'Not run yet'})
_r3 = globals().get('r3', {'ok': False, 'seconds': 0.0, 'final': 'Not run yet'})
_r_fail_a = globals().get('r_fail_a', {'ok': False, 'seconds': 0.0, 'final': 'Not run yet'})
_r_fail_c = globals().get('r_fail_c', {'ok': False, 'seconds': 0.0, 'final': 'Not run yet'})

rows = [
    ("Test 1 – Tech fest", _r1),
    ("Test 2 – Guest speaker", _r2),
    ("Test 3 – Cultural day", _r3),
    ("Failure A – incomplete", _r_fail_a),
    ("Failure C – API outage", _r_fail_c)
]

print(f"{'Run':28} {'OK':5} {'Seconds':8} {'Final answer length (chars)':>28}")
for name, r in rows:
    final_len = len(r['final']) if r.get('final') else 0
    print(f"{name:28} {str(r.get('ok', False)):5} {r.get('seconds', 0.0):<8} {final_len:>28}")

Run                          OK    Seconds   Final answer length (chars)
Test 1 – Tech fest           True  14.8                             4100
Test 2 – Guest speaker       True  15.4                             3691
Test 3 – Cultural day        False 65.3                              307
Failure A – incomplete       True  76.8                             3903
Failure C – API outage       False 5.5                               293


In [ ]:
import logging

logging.getLogger("crewai").setLevel(logging.ERROR)
logging.getLogger("crewai.llms.base_llm").setLevel(logging.ERROR)

## 10. Debug Log
Record **genuine** problems you met while building and running *your* notebook. Below are the entries structured exactly as the assignment requires; they describe issues that commonly occur with this exact stack. **Edit each entry so it reflects what you actually saw on your own runs** (change the error text/solution if yours differed, delete any you did not hit, and add your own).

| # | Error / issue encountered | How the cause was identified | Solution implemented | Remaining limitation |
|---|---|---|---|---|
| 1 | Agents sometimes **skipped a tool** (e.g. never called `budget_calculator`) and wrote a plausible-looking number instead. | Compared the final answer with `TOOL_LOG`; the coverage line showed `not used: ['budget_calculator']`. | Rewrote task descriptions to say *"You MUST call these tools"* with a numbered list, added `expected_output` that says "(from tool)", and added the TOOL COVERAGE check in `show_run`. | A weaker LLM can still ignore instructions; the check detects it but does not force the call. |
| 2 | Tool failed with **`ValueError` / bad arguments** when the LLM passed a date like `25 Oct 2026` instead of `YYYY-MM-DD`, and an exception inside a tool stopped the task. | The traceback pointed to `strptime`; printing tool arguments in `TOOL_LOG` showed the wrong format. | Wrapped every tool in `try/except`, returned a readable `ERROR: date must be YYYY-MM-DD` message so the agent retries with the right format; stated the format in each docstring and task. | Relies on the agent retrying; max_iter limits the number of retries. |
| 3 | **Open-Meteo returned HTTP 400** for event dates more than 16 days away, so the weather tool showed an error. | Printed `r.status_code` and `r.text` in the API test cell. | Added a fallback in `weather_forecast`: on non-200, fetch the next 3 days and tell the reader the event date is outside the forecast window. | Weather for far-future dates is only indicative, not a real forecast. |
| 4 | `ModuleNotFoundError: google.colab` / secrets returned `None` → Nexus authentication error. | Error message and printing `bool(NEXUS_BASE_URL)`. | Added the Nexus connectivity test cell and the setup note about enabling *Notebook access* for each secret. | Notebook needs Colab; cannot run locally without edits. |

*(Add your own entries here if you hit other problems – e.g. CrewAI version warnings, rate limits, memory/embedding errors.)*

In [ ]:
# Interactive mode: type any campus-event request and the 4 agents handle it
while True:
    user_req = input("\nDescribe your campus event request (or type 'quit'): ")
    if user_req.strip().lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    show_run("Interactive request", user_req)


Describe your campus event request (or type 'quit'): Plan a one-day inter-college technical fest at NITTE Mangaluru on 2026-11-06 for about 400 participants. Include a hackathon, paper presentation and robo race. Two judges are visiting, one arriving from Bengaluru. Also prepare a student announcement and a preparation checklist.
TEST: Interactive request
INPUT REQUEST:
  Plan a one-day inter-college technical fest at NITTE Mangaluru on 2026-11-06 for about 400 participants. Include a hackathon, paper presentation and robo race. Two judges are visiting, one arriving from Bengaluru. Also prepare a student announcement and a preparation checklist. 

-- AGENT STEPS (each task result is passed as context to the next agent) --
 1. Event Planner: Here is the complete plan for the one-day inter-college technical fest at NITTE Mangaluru:  **Assumptions:** - Event Date: 2026-11-06 (provided directly by the user) - Expected Attendees: 400 (provide...
 2. Event Planner: Here is the complete plan